# Final Assessment — scRNA-seq on PBMC68k

## Bioinformatics for HCT — Take-home final

**Instructor:** Norah AlGhamdi · **Course:** Introduction to Bioinformatics

---

## Instructions

Apply everything you learned in Day 7 to a **new dataset**: PBMC68k_reduced (700 cells, pre-processed). You have **5 graded tasks** for a total of **100 points**.

**How to submit:**
- Complete all 5 tasks in this notebook (fill in the code cells and answer the questions)
- Save your completed `.ipynb` file (File → Download → Download .ipynb)
- Email it to Norah by **next Friday**
- Optionally include a PDF export with figures embedded

**Grading rubric:**

| Task | Points |
|---|---|
| 1. QC and filter | 20 |
| 2. Normalize + dimensional reduction + UMAP | 20 |
| 3. Clustering at multiple resolutions | 20 |
| 4. Marker genes + cell type annotation | 20 |
| 5. Open-ended biological finding | 20 |
| **Total** | **100** |

Good luck — and have fun! 🧬

---

## Setup

Run this cell first. It installs Scanpy and loads the PBMC68k_reduced dataset (already preprocessed — comes with UMAP, clusters, and marker genes already in the AnnData).

In [ ]:
!pip install scanpy leidenalg --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.set_figure_params(dpi=80, facecolor='white')
print('Scanpy:', sc.__version__)

In [ ]:
# Load the dataset — Scanpy ships with it
adata = sc.datasets.pbmc68k_reduced()
adata

**⚠️ Important note about this dataset:**

`pbmc68k_reduced` is a **pre-processed subset** of the PBMC68k data — it has 700 cells × 765 genes and the entire upstream pipeline (PCA, UMAP, clustering) has already been run. The `.X` is the log-normalized expression, not raw counts.

**This means a few things:**
- You CAN'T run `normalize_total` again (data is already normalized)
- You CAN re-run clustering, marker gene identification, and visualization
- You CAN do QC analysis on the existing metadata fields
- Look at `adata.obs.columns` to see what's already there

In [ ]:
print('Cells x Genes:', adata.shape)
print()
print('What metadata is already in adata.obs:')
print(adata.obs.columns.tolist())
print()
print('Existing UMAP / PCA:', list(adata.obsm.keys()))

---
# 📝 Task 1 — Quality control (20 points)

Examine the QC metrics already in this dataset and discuss what you see.

**To do:**
1. Plot violin plots of `n_genes`, `percent_mito`, and `n_counts` (10 pts)
2. Plot the scatter of `n_counts` vs `percent_mito` (5 pts)
3. **Answer in markdown:** Is this dataset already well-filtered? What evidence supports your answer? (5 pts)

In [ ]:
# Task 1.1 — Violin plot of QC metrics
# Hint: use sc.pl.violin with the keys from adata.obs

# YOUR CODE HERE


In [ ]:
# Task 1.2 — Scatter plot
# Hint: use sc.pl.scatter(adata, x=..., y=...)

# YOUR CODE HERE


### Task 1.3 — Your written answer

*(Replace this text with your answer. 2-4 sentences.)*

Is this dataset already well-filtered? What evidence supports your answer?

---
# 📝 Task 2 — Visualize the existing UMAP (20 points)

The dataset already has UMAP coordinates. Use them to explore the data.

**To do:**
1. Plot the UMAP colored by `bulk_labels` (the original 10x cell type labels) (5 pts)
2. Plot the UMAP colored by `n_counts` (5 pts)
3. Plot the UMAP colored by `phase` (cell cycle phase) (5 pts)
4. **Answer in markdown:** Does cell cycle phase affect the clustering? (5 pts)

In [ ]:
# Task 2 — UMAPs
# Hint: sc.pl.umap(adata, color='bulk_labels')

# YOUR CODE HERE


### Task 2.4 — Your written answer

*(Replace this text with your answer.)*

Does cell cycle phase appear to drive any of the cluster structure?

---
# 📝 Task 3 — Re-cluster at multiple resolutions (20 points)

Run Leiden clustering at 3 different resolutions and compare.

**To do:**
1. Run Leiden at resolution = 0.3, 0.5, and 1.0 (with `key_added` to save each separately) (10 pts)
2. Plot all 3 side by side on UMAP (5 pts)
3. **Answer in markdown:** Which resolution gives the most biologically meaningful result, and why? (5 pts)

In [ ]:
# Task 3.1 — Run Leiden at 3 resolutions
# Hint:
# sc.tl.leiden(adata, resolution=0.3, key_added='leiden_low', flavor='igraph', n_iterations=2, directed=False)

# YOUR CODE HERE


In [ ]:
# Task 3.2 — Plot all 3 side by side
# Hint: sc.pl.umap(adata, color=['leiden_low','leiden_med','leiden_high'], ncols=3)

# YOUR CODE HERE


### Task 3.3 — Your written answer

*(Replace this text with your answer.)*

Which resolution did you pick and why? What did the other resolutions get wrong?

---
# 📝 Task 4 — Marker genes + cell type annotation (20 points)

Find marker genes per cluster and annotate at least 3 cell types.

**To do:**
1. Run `sc.tl.rank_genes_groups` on your chosen Leiden clustering (5 pts)
2. Plot the top markers per cluster as a dot plot (5 pts)
3. Annotate at least 3 clusters by matching markers to known PBMC types (10 pts)

**Use these reference markers:**

| Cell type | Markers |
|---|---|
| T cell | CD3D, CD3E, IL7R |
| B cell | MS4A1, CD79A |
| Monocyte CD14 | CD14, LYZ |
| Monocyte CD16 | FCGR3A |
| NK | GNLY, NKG7 |
| Dendritic | FCER1A, CST3 |

In [ ]:
# Task 4.1 — Find marker genes
# Hint: sc.tl.rank_genes_groups(adata, 'leiden_med', method='wilcoxon')

# YOUR CODE HERE


In [ ]:
# Task 4.2 — Dotplot of markers
# Hint: define a marker dict, then sc.pl.dotplot(adata, marker_genes, groupby='leiden_med')

# YOUR CODE HERE


In [ ]:
# Task 4.3 — Map clusters to cell types
# Hint:
# cluster_to_celltype = {'0': 'T cell', '1': 'B cell', ...}
# adata.obs['my_annotation'] = adata.obs['leiden_med'].map(cluster_to_celltype).astype('category')
# sc.pl.umap(adata, color='my_annotation', legend_loc='on data')

# YOUR CODE HERE


---
# 📝 Task 5 — Open-ended biological finding (20 points)

**Find ONE interesting biological observation in this dataset** and write a short conclusion.

Possible angles:
- **A rare cell type** — find a cluster with <5% of cells. What is it? Why is it interesting?
- **Sex/condition differences** — are there metadata fields that split clusters?
- **A surprising marker** — a gene you didn't expect at the top of a cluster's marker list
- **A specific gene's expression pattern** — pick any gene and explore where it's expressed
- **Comparing your annotations to `bulk_labels`** — where do you disagree with the original?

**Required deliverable:**
1. At least one figure supporting your finding (5 pts)
2. Code to reproduce your finding (5 pts)
3. **One paragraph (4-6 sentences)** explaining what you found, why it's interesting, and what you'd test next (10 pts)

In [ ]:
# Task 5 — Your exploration
# Make plots, run analyses, find something interesting!

# YOUR CODE HERE



### Task 5 — Your written conclusion

*(Replace this text with your 4-6 sentence paragraph.)*

**What did you find?** (e.g. "A rare cluster of FCER1A+ cells likely represents plasmacytoid dendritic cells...")

**Why is it interesting biologically?**

**What would you do next if you had more time?**

---
## Submission checklist

Before submitting, verify:

- [ ] All 5 tasks have code that **runs without errors**
- [ ] All 5 tasks have **written answers** (not left as placeholder text)
- [ ] Plots are visible in the notebook (don't just save them — display them too)
- [ ] File saved with your name in the filename: `Day7_Final_<YourName>.ipynb`
- [ ] Emailed to `nalghamd88@gmail.com` by **next Friday**

**Bonus points** for:
- Clean, readable code
- Thoughtful written answers
- Creative biological interpretation in Task 5

Good luck! 🎯